In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)


from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import *
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.Files import filename_to_dataframe
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *
from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrames

In [ ]:
## Check keys in each file
optimization_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/old2/cc1pi_1e20_training.df"
test_file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/old2/plane2/cc1pi_5e18_CV.df"
print("keys in test_file")
splh.print_keys(optimization_file)

## Check split multiplicity
print("mc_bnb_cosmic_file n_split: %d" %splh.get_n_split(optimization_file))

In [ ]:
## Define keys to load
print('MC dataframes')
n_max_concat = 12 ## for big files, each key could have more than one split
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
optimization_df = splh.load_dfs(optimization_file, keys2load, n_max_concat)
test_df = splh.load_dfs(test_file, keys2load, n_max_concat)

print('test data loaded!')

# Filter DataFrame

In [ ]:
#Perform duplication validation
print("duplication for Spring Production BNB + Cosmic sample")
DFCleaning.find_duplicate_run_evt_combinations(optimization_df['hdr'])
DFCleaning.find_duplicate_run_evt_combinations(test_df['hdr'])

In [ ]:
DFCleaning.plot_duplicate_run_subrun_evt_distribution(optimization_df["hdr"], "optimization_df")
DFCleaning.plot_duplicate_run_subrun_evt_distribution(test_df["hdr"], "test_df")

In [ ]:
### Filter the hdr DataFrame first, then filter other DataFrames by matching with the hdr DataFrame
optimization_df["hdr"] = DFCleaning.filter_unique_events(optimization_df["hdr"])
DFCleaning.find_duplicate_run_evt_combinations(optimization_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    optimization_df[key] = DFCleaning.filter_using_hdr(optimization_df[key], optimization_df["hdr"])

test_df["hdr"] = DFCleaning.filter_unique_events(test_df["hdr"])
DFCleaning.find_duplicate_run_evt_combinations(test_df["hdr"])
#filter the rest of dataframe keys
for key in keys2load:
    if key == "hdr":
        continue
    test_df[key] = DFCleaning.filter_using_hdr(test_df[key], test_df["hdr"])

# Check normalization

In [ ]:
def get_n_evt(df):
    unique_count = df.index.droplevel(
        list(df.index.names[2:])  # drop everything except first two levels
    ).nunique()
    return unique_count

In [ ]:
## Collect pot scale for MC
mc_tot_pot = optimization_df["hdr"]['pot'].sum()
#mc_low_th_tot_pot = mc_rockbox_th1to100_dfs["hdr"]['pot'].sum()

data_tot_pot = 3.63462e+18
#data_tot_pot = data_bnb_light_dfs["hdr"]['pot'].sum()
#data_tot_TOR860 = data_bnb_light_dfs["pot"]['TOR860'].sum()
#data_tot_TOR875 = data_bnb_light_dfs["pot"]['TOR875'].sum()

print("mc_tot_pot: %e" %(mc_tot_pot))
#print("mc_low_thtot_pot: %e" %(mc_low_th_tot_pot))

#print("data_tot_pot: %e" %(data_tot_pot))
#print("data_tot_TOR860: %e" %(data_tot_TOR860))
#print("data_tot_TOR875: %e" %(data_tot_TOR875))

target_pot = data_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
#mc_low_th_scale = target_pot / mc_low_th_tot_pot
print("MC POT scale: %.3f" %(mc_pot_scale))
#print("MC Low Th. POT scale: %.3f" %(mc_low_th_scale))

# Do truth matching

In [ ]:
evt_df = optimization_df['cc1pi']
hdr_df = optimization_df['hdr']
nu_df = optimization_df['nudf']

test_evt_df = test_df['cc1pi']
test_hdr_df = test_df['hdr']
test_nu_df = test_df['nudf']

In [ ]:
new_columns = []
for c in nu_df.columns:
    new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
nu_df.columns = pd.MultiIndex.from_tuples(new_columns)

new_columns_test_df = []
for c in test_nu_df.columns:
    new_columns_test_df.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
test_nu_df.columns = pd.MultiIndex.from_tuples(new_columns_test_df)

In [ ]:
matchdf = ph.multicol_merge(evt_df.reset_index(), nu_df.reset_index(),
                            left_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("slc", "tmatch","idx","","","")],
                            right_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("rec.mc.nu..index", "","","","","")], 
                            how="left") ## -- save all sllices
#Reindex so it is again "__ntuple","entry", "slice_id"
matchdf = matchdf.set_index(evt_df.index.names, verify_integrity=True)
#Remove "rec.mc.nu..index"
matchdf = matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])
matchdf.loc[:, ('truth', 'nu_categ','','','','')] = (
    matchdf.loc[:, ('truth', 'nu_categ','','','','')].fillna('cosmic')
)
evt_df = matchdf

test_matchdf = ph.multicol_merge(test_evt_df.reset_index(), test_nu_df.reset_index(),
                            left_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("slc", "tmatch","idx","","","")],
                            right_on=[("__ntuple", "","","","",""),("entry", "","","","",""), ("rec.mc.nu..index", "","","","","")], 
                            how="left") ## -- save all sllices
#Reindex so it is again "__ntuple","entry", "slice_id"
test_matchdf = test_matchdf.set_index(test_evt_df.index.names, verify_integrity=True)
#Remove "rec.mc.nu..index"
test_matchdf = test_matchdf.drop(columns=[('rec.mc.nu..index','','','','','')])
test_matchdf.loc[:, ('truth', 'nu_categ','','','','')] = (
    test_matchdf.loc[:, ('truth', 'nu_categ','','','','')].fillna('cosmic')
)
test_evt_df = test_matchdf

# Optimize cosmic removal

# BC Flash matcher

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/Graphs/Optimization"
os.makedirs(file_dir, exist_ok=True)  # create directory if needed

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)

mask = (evt_df.slc.cut.obvious_cosmic == True) &  (evt_df.slc.cut.inside_FV == True) 
evt_df_reset = evt_df[mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)

# Drop duplicates based on the remaining index levels
evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]

# Split signal and background
signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]

column = ('slc', 'barycenterFM', 'score', '', '', '')

df_opt, best, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=('slc', 'barycenterFM', 'score', '', '', ''),
    cut_type=">",
    xlabel="Barycenter FM score",
    title="ν selection optimization",
    signal_name=r"$\nu$",
    bkg_name="Cosmic background",
    xlim=(0.002, 0.2),
    nbins=25,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
)

fig.savefig(file_dir + "/bfm_optimization.png", dpi=300)
plt.show()

# Nu Score

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)
mask = (evt_df.slc.cut.obvious_cosmic == True) &  (evt_df.slc.cut.inside_FV == True) 
evt_df_reset = evt_df[mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)

# Drop duplicates based on the remaining index levels
evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]

# Split signal and background
#signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ == "CC1pi")]
#bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ != "CC1pi")]
signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]

column = ('slc', 'nu_score', '', '', '', '')

df_opt, best,fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=column,
    cut_type=">",
    xlabel=r"$\nu$ Score",
    title="ν selection optimization",
    signal_name=r"$\nu$",
    bkg_name="cosmic",
    xlim=(0, 1),
    nbins=50,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
)

fig.savefig(file_dir + "/nu_score_optimization_neutrinos.png", dpi=300)
plt.show()

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)
mask = (
    (evt_df.slc.cut.inside_FV == True)& (evt_df.slc.cut.obvious_cosmic ==  True)
    #&(evt_df.slc.cut.t0 == True) 
    #& (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    #&  (evt_df.slc.cut.containment == True) 
    #& (evt_df.slc.cut.angle == True) 
    #& (evt_df.slc.cut.proton_BDT == True)
    #& (evt_df.slc.cut.michel == True) 
    #& (evt_df.slc.cut.extra_pion == True) 
   
   
)


evt_df_reset = evt_df[mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)

# Drop duplicates based on the remaining index levels
evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]

# Split signal and background
signal_df = evt_df_unique[ ((evt_df_unique.truth.nu_categ == "CC1pi") | (evt_df_unique.truth.nu_categ == "other_CC1pi"))]
bkg_df    = evt_df_unique[ ((evt_df_unique.truth.nu_categ != "CC1pi") & (evt_df_unique.truth.nu_categ != "other_CC1pi"))]

#signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
#bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]

column = ('slc', 'nu_score', '', '', '', '')

df_opt, best,fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=column,
    cut_type=">",
    xlabel=r"$\nu$ score",
    title="ν selection optimization",
    signal_name=r"$\nu_{\mu}CC1\pi$",
    bkg_name="background",
    xlim=(0, 1),
    nbins=50,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
)


fig.savefig(file_dir + "/nu_score_optimization_cc1pi.png", dpi=300)
plt.show()

# Chi2 cut

In [ ]:
SLICE_LEVELS = ["__ntuple", "entry", "rec.slc..index"]
chi2_dir = "/exp/sbnd/data/users/lpelegri/Graphs/Optimization/Chi2"
os.makedirs(chi2_dir, exist_ok=True)  # create directory if needed

In [ ]:
# -------------------------
# Select signal and background
# -------------------------
cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True)

signal_df = evt_df[ cut_mask&
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[ cut_mask&
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]



# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]

# -------------------------
# Manual 2D cut scan
# -------------------------
best_2d, results_df = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_chi2_mu, col_chi2_p],
    cut_sign=["<", ">"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[0, 0],               # min scan range
    cut_max=[60, 300],            # max scan range
    n_steps= [61,101],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)


# -------------------------
# Manual 2D cut scan
# -------------------------
best3d, results_df3d = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_chi2_mu, col_chi2_p,col_len],
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[30, 70,3],               # min scan range
    cut_max=[50, 120,20],            # max scan range
    n_steps= [21,51,18],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)


In [ ]:
# Split signal and background

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True)

signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < best_2d["cuts"][0]) & (signal_df[col_chi2_p] > best_2d["cuts"][1])]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < best_2d["cuts"][0]) & (bkg_df[col_chi2_p] > best_2d["cuts"][1])]

df_opt_len, best_len,fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_len,
    cut_type=">",
    xlabel="length",
    title="pfp optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="Other",
    xlim=(0, 30),
    nbins=31,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0 # <-- new parameter
)
fig.savefig(chi2_dir + "/lenop_plus_2d.png", dpi=300)
plt.show()

In [ ]:

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True)

signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]
OptimizationUtils.plot_2d_cut_individual_manual(
    signal_df,
    bkg_df,
    col1=col_chi2_mu,
    col2=col_chi2_p,
    cutnum1=0,
    cutnum2=1,
    best=best_2d,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    signal_label="Signal",
    bkg_label="Background",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap
)

OptimizationUtils.plot_2d_cut_metric_heatmap(
    signal_df,
    bkg_df,
    cutnum1=0,
    cutnum2=1,
    best=best_2d,
    results_df=results_df,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    signal_label="Signal",
    bkg_label="Background",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap
)

In [ ]:
# Split signal and background

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True)

signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < 20) & (signal_df[col_len] > 10)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < 20) & (bkg_df[col_len] > 10)]

df_opt_len, best_chi2,fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_chi2_p,
    cut_type=">",
    xlabel=r"$\chi^2_p$",
    title="pfp optimization",
    signal_name=r"$\mu / \pi$",
    bkg_name="Other",
    xlim=(0, 300),
    nbins=51,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
    min_pur=0.0 # <-- new parameter
)
fig.savefig(chi2_dir + "/fixedlen_fixedchi2mu_chi2pop.png", dpi=300)
plt.show()

In [ ]:

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True)
signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) != 211) & (abs(evt_df.pfp.trk.truth.p.pdg) != 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

print("Fixed 85")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_chi2_mu, col_chi2_p,col_len],
    cuts=[20,85,10],
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
)
fig = ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = [r"$\mu/\pi$","other"],
    y_labels = [r"$\mu/\pi$","other"]
)
fig.savefig(chi2_dir + "/confusion_matrix_def.png", dpi=300)
plt.show()

best_fixed = {"cuts": [20,85]}
OptimizationUtils.plot_2d_cut_individual_manual(
    signal_df,
    bkg_df,
    col1=col_chi2_mu,
    col2=col_chi2_p,
    cutnum1=0,
    cutnum2=1,
    save_dir= chi2_dir + "/chi2mu_chi2p_",
    best=best_fixed,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    signal_label=r"$\mu/\pi$",
    bkg_label="other",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap
)

print("3D OPT")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_chi2_mu, col_chi2_p,col_len],
    cuts=best3d["cuts"],
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
)
fig =  ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = [r"$\mu/\pi$","other"],
    y_labels = [r"$\mu/\pi$","other"]
)
fig.savefig(chi2_dir + "/confusion_matrix_3dop.png", dpi=300)
plt.show()


print("2D OPT + Len")
print(np.append(best_2d["cuts"], best_len["cut"]))
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_chi2_mu, col_chi2_p,col_len],
    cuts = np.append(best_2d["cuts"], best_len["cut"]),
    cut_sign=["<", ">", ">"],          # "<" for chi2_mu, ">" for chi2_proton
)
fig =  ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = [r"$\mu/\pi$","other"],
    y_labels = [r"$\mu/\pi$","other"]
)
fig.savefig(chi2_dir + "/confusion_matrix_2dop_len.png", dpi=300)
plt.show()


In [ ]:
def plot_2d_by_category(
    df,
    col1,
    col2,
    category_col,
    best,
    cutnum1,
    cutnum2,
    save_dir=None,
    xlabel="Variable 1",
    ylabel="Variable 2",
    xlim=None,
    ylim=None,
    bins=50,
    cmap="plasma"
):
    """
    Groups a DataFrame by 'category_col', produces a 2D histogram 
    for every unique category, including a 'MIP region' label,
    and saves them individually.
    """
    import os
    import matplotlib.pyplot as plt
    import numpy as np

    # 1. Setup Directory
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 2. Extract Cut values from 'best' dict
    cut1_best = best["cuts"][cutnum1]
    cut2_best = best["cuts"][cutnum2]
    hist_range = [xlim, ylim] if xlim and ylim else None

    # 3. Get unique categories, sorted
    categories = sorted(df[category_col].unique())
    print(f"Found categories: {categories}")

    # 4. Iterate and Plot
    for cat in categories:
        # Filter data for this specific category
        cat_df = df[df[category_col] == cat].loc[:, [col1, col2]].dropna()
        
        if cat_df.empty:
            print(f"Skipping {cat}: No data after dropna.")
            continue

        fig, ax = plt.subplots(figsize=(7, 6))
        
        # Create the 2D Histogram
        im = ax.hist2d(
            cat_df[col1], 
            cat_df[col2], 
            bins=bins, 
            range=hist_range, 
            cmap=cmap
        )
        
        # Add colorbar
        plt.colorbar(im[3], ax=ax, label="Events")
        
        # --- NEW: MIP REGION LABEL ---
        # (0.02, 0.95) places it at 2% from the left, 95% from the bottom of the axes.
        '''
        I’m ax.text(
            0.02, 0.98, 
            "MIP region", 
            transform=ax.transAxes, 
            fontsize=12, 
            fontweight='bold', 
            color='black', # Changed to white for better contrast with plasma cmap
            va='top', 
            bbox=dict(facecolor='white', edgecolor='black') # Added subtle background box
        )
        '''
        
        # Plot cut lines
        ax.axvline(cut1_best, color="black", linestyle="--", linewidth=2.0)
        ax.axhline(cut2_best, color="black", linestyle="--", linewidth=2.0)
        
        # Labels and Title
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{cat}")
        plt.tight_layout()

        # 5. Save Logic
        if save_dir:
            # Clean category name for filename
            clean_name = str(cat).replace(" ", "_").replace("/", "_")
            filename = f"hist2d_MIP_{clean_name}.pdf"
            path = os.path.join(save_dir, filename)
            plt.savefig(path, format='pdf', dpi=300)
            print(f"Saved: {path}")
        
        plt.show()

    # --- Total Plot (Combined) ---
    fig_tot, ax_tot = plt.subplots(figsize=(7, 6))
    total_df = df.loc[:, [col1, col2]].dropna()
    im_tot = ax_tot.hist2d(total_df[col1], total_df[col2], bins=bins, range=hist_range, cmap=cmap)
    plt.colorbar(im_tot[3], ax=ax_tot, label="Total Events")
    
    # Add MIP region label to the total plot too
    '''
    ax_tot.text(0.02, 0.98, "MIP region", transform=ax_tot.transAxes, 
                fontsize=12, fontweight='bold', color='black', va='top', 
                bbox=dict(facecolor='white', alpha=0.3, edgecolor='black'))
    '''
    
    ax_tot.axvline(cut1_best, color="black", linestyle="--", linewidth=2.0)
    ax_tot.axhline(cut2_best, color="black", linestyle="--", linewidth=2.0)
    ax_tot.set_xlabel(xlabel)
    ax_tot.set_ylabel(ylabel)
    ax_tot.set_title("Total (All Categories Combined)")
    plt.tight_layout()
    
    if save_dir:
        plt.savefig(os.path.join(save_dir, "hist2d_total_combined.pdf"), format='pdf', dpi=300)
    
    plt.show()

In [ ]:
cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True)
df = evt_df[cut_mask &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
    & (evt_df.pfp.is_exiting == False)
]

pdg_col = ('pfp', 'trk', 'truth', 'p', 'pdg', '')
endp_col = ('pfp', 'trk', 'truth', 'p', 'end_process', '')
p_type_col =  ('pfp', 'trk', 'truth', 'p', 'p_type', '')

print(df.loc[(abs(df.pfp.trk.truth.p.pdg) == 2212), endp_col])
df.loc[(abs(df.pfp.trk.truth.p.pdg) == 2212) & (df.pfp.trk.truth.p.end_process == 7),p_type_col] = "inelastic proton"
df.loc[(abs(df.pfp.trk.truth.p.pdg) == 2212) & (df.pfp.trk.truth.p.end_process == 45),p_type_col] = "stopping proton"

dir_p = "/exp/sbnd/data/users/lpelegri/Graphs/ProtonBDT"
os.makedirs(dir_p, exist_ok=True)  # create directory if needed
plot_2d_by_category(
    df,
    col1=col_chi2_mu,
    col2=col_chi2_p,
    category_col = p_type_col,
    cutnum1=0,
    cutnum2=1,
    save_dir= dir_p + "/chi2mu_chi2p",
    best=best_fixed,
    xlabel=r"$\chi^2_\mu$",
    ylabel=r"$\chi^2_p$",
    xlim=(0, 60),
    ylim=(0, 300),
    bins=60,
    cmap= sunset_cmap
)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
from matplotlib.colors import to_rgba
import os


cut_mask = (test_evt_df.slc.cut.nu_score == True) & (test_evt_df.slc.cut.t0 == True) & (test_evt_df.slc.cut.track == True) & (test_evt_df.slc.cut.inside_FV == True)
df_plot = test_evt_df[cut_mask &
    (abs(test_evt_df.pfp.trk.len) > 10) & (abs(test_evt_df.pfp.trackScore) > 0.5) & (test_evt_df.pfp.dist_to_vertex < 10)
    & (test_evt_df[col_chi2_mu] < 20) & (test_evt_df[col_chi2_p] > 85)].copy()

# --- 2. Configuration & Column Keys ---
p_type_col = ('pfp', 'trk', 'truth', 'p', 'p_type', '')
px_col, py_col, pz_col = [('pfp', 'trk', 'truth', 'p', 'genp', axis) for axis in ['x', 'y', 'z']]
weight_col = ('slc', 'wgt', '', '', '', '') 

category_colors_pfp = {
    "muon": "#1f77b4", "stopping pion": "#d62728", "inelastic pion": "#2ca02c",
    "proton": "#ff7f0e", "other": "#7f7f7f", "shower": "#e377c2",
}

# --- 3. Pre-processing ---
df_plot.loc[(abs(df_plot.pfp.trk.truth.p.pdg) == 2212), p_type_col] = "proton"
df_plot['true_momentum'] = np.sqrt(df_plot[px_col]**2 + df_plot[py_col]**2 + df_plot[pz_col]**2)

# --- 4. Plotting ---
fig, ax = plt.subplots(figsize=(8, 6))
unique_cats = sorted(df_plot[p_type_col].dropna().unique())
bins = np.linspace(0, 5, 51)

# Modern, warning-free colormap access
import matplotlib as mpl
fallback_colors = mpl.colormaps['tab10'].resampled(len(unique_cats))

mc_hand = []

for i, cat in enumerate(unique_cats):
    # Accessing the color remains the same
    plot_color = category_colors_pfp.get(cat, fallback_colors(i))
    
    # ... (rest of your plotting logic) ...
    mask = df_plot[p_type_col] == cat
    data = df_plot.loc[mask, 'true_momentum']
    weights = df_plot.loc[mask, weight_col] if weight_col in df_plot.columns else np.ones_like(data)
    
    if data.empty:
        continue

    plot_color = category_colors_pfp.get(cat, fallback_colors(i))

    # Plot histogram
    ax.hist(data, bins=bins, stacked=False, weights=weights,
                histtype='stepfilled', color=plot_color, alpha=0.3)
    ax.hist(data, bins=bins, stacked=False, weights=weights,
                histtype='step', color=plot_color, linewidth=2)
    

    # --- 5. LEGEND PATCHES ---
    # Creates a patch with a shaded face and a solid edge to match the plot style
    mc_hand.append(Patch(
        facecolor=to_rgba(plot_color, 0.3), 
        edgecolor=plot_color, 
        label=str(cat).replace("_", " ") # Clean up label strings if needed
    ))

# --- 6. Styling & Manual Legend ---
ax.set_title("", fontsize=14, loc='center')
ax.set_xlabel("True Momentum [GeV]", fontweight='bold', loc='right')
ax.set_ylabel("PFPs", fontweight='bold', loc='top')
ax.set_xlim(0, 5)
ax.set_ylim(bottom=0)
ax.tick_params(direction='in', top=True, right=True, which='both', length=6)

# Pass the custom patches to the legend
ax.legend(handles=mc_hand, frameon=False, fontsize=20, loc='upper right')

plt.tight_layout()

# --- 7. Save ---
save_folder = "/exp/sbnd/data/users/lpelegri/Graphs/ProtonBDT"
os.makedirs(save_folder, exist_ok=True)
plt.savefig(os.path.join(save_folder, "momentum_ptype_breakdown.pdf"), format='pdf', bbox_inches='tight')

plt.show()

# Shower cut

In [ ]:
def shower_cut_mask(df, group_levels, min_shower_ke = 0):
    is_pandora_primary_mask = (df.pfp.parent_is_primary == True)
    is_shower_mask = (df.pfp.trackScore >= 0) & (df.pfp.trackScore < CTE.max_shower_track_score)
    energy_mask = (df.pfp.shw.bestplane_energy > min_shower_ke)
    shower_df = df[is_pandora_primary_mask & is_shower_mask & energy_mask] 
        
    # Count how many pfps per slice
    shower_counts = shower_df.groupby(level=group_levels).size()
    
    # Get only slices with at least 2 pfps
    non_valid_slices = shower_counts[(shower_counts > 0)].index

    # Apply the mask to original DataFrame
    final_mask = pd.Series(~df.index.droplevel('rec.slc.reco.pfp..index').isin(non_valid_slices), index=df.index)
   
    return final_mask

In [ ]:
# Drop the last index level (rec.slc.reco.pfp..index)
# Split signal and background
#cut_mask = (evt_df.slc.nu_score > 0.55) & (evt_df.slc.barycenterFM.score > 0.03)

cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.inside_FV == True)  & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True)

#cut_mask = True
signal_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 11) & (abs(evt_df.pfp.trackScore) < 0.5) & (evt_df.pfp.parent_is_primary == True))
]

bkg_df = evt_df[cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 22) & (abs(evt_df.pfp.trackScore) < 0.5) & (evt_df.pfp.parent_is_primary == True))
]

df_opt, best, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=('pfp', 'shw', 'bestplane_energy', '', '', ''),
    cut_type="<",
    xlabel="best plane energy [GeV]",
    title="ν selection optimization",
    signal_name=r"e",
    bkg_name=r"$\gamma$",
    xlim=(0, 0.2),
    nbins=25,
    legend_loc="upper right",
    cut_unit="",
    normalize_hist = False,
)

In [ ]:

cut_mask = (test_evt_df.slc.cut.nu_score == True) & (test_evt_df.slc.cut.t0 == True) & (test_evt_df.slc.cut.inside_FV == True)  & (test_evt_df.slc.cut.track == True)  & (test_evt_df.slc.cut.MIP_candidates == True)

print("Control")
HelperFunctions.print_category_metrics(test_evt_df, test_evt_df[cut_mask], target_categ="CC1pi")
print(0.065)
HelperFunctions.print_category_metrics(test_evt_df, test_evt_df[cut_mask & shower_cut_mask(test_evt_df, SLICE_LEVELS, min_shower_ke = 0.06)], target_categ="CC1pi")
print(0.06)
HelperFunctions.print_category_metrics(test_evt_df, test_evt_df[cut_mask & shower_cut_mask(test_evt_df, SLICE_LEVELS, min_shower_ke = 0.06)], target_categ="CC1pi")
print(0.055)
HelperFunctions.print_category_metrics(test_evt_df, test_evt_df[cut_mask & shower_cut_mask(test_evt_df, SLICE_LEVELS, min_shower_ke = 0.055)], target_categ="CC1pi")
print(0.045)
HelperFunctions.print_category_metrics(test_evt_df, test_evt_df[cut_mask & shower_cut_mask(test_evt_df, SLICE_LEVELS, min_shower_ke = 0.045)], target_categ="CC1pi")

# Angle Cut 

In [ ]:
#cut_mask = (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) & (evt_df.slc.cut.inside_FV == True)  & (evt_df.slc.cut.track == True)  & (evt_df.slc.cut.MIP_candidates == True)

cut_mask = (
    #(evt_df.slc.cut.nu_score == True) 
     (evt_df.slc.cut.inside_FV == True) & (evt_df.slc.cut.t0 == True)
    & (evt_df.slc.cut.track == True) 
    & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
)


# Drop the last index level (rec.slc.reco.pfp..index)
evt_df_reset = evt_df[cut_mask].reset_index(level='rec.slc.reco.pfp..index', drop=True)

# Drop duplicates based on the remaining index levels
evt_df_unique = evt_df_reset[~evt_df_reset.index.duplicated(keep='first')]


# Split signal and background
signal_df = evt_df_unique[
    (evt_df_unique.truth.nu_categ == "CC1pi") |
    (evt_df_unique.truth.nu_categ == "other_CC1pi")
]
bkg_df    = evt_df_unique[((evt_df_unique.truth.nu_categ != "CC1pi") & (evt_df_unique.truth.nu_categ != "other_CC1pi"))]
#signal_df = evt_df_unique[(evt_df_unique.truth.nu_categ != "cosmic")]
#bkg_df    = evt_df_unique[(evt_df_unique.truth.nu_categ == "cosmic")]

column = ('slc', 'measure_var', 'angle_between_candidates', '', '', '')

df_opt, best, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column=column,
    cut_type="<",
    xlabel="Angle between candidates [rad]",
    title="ν selection optimization",
    signal_name=r"$\nu_{\mu}CC1\pi$",
    bkg_name="background",
    xlim=(0, 3.2),
    nbins=40,
    legend_loc="upper right",
    cut_unit="[rad]",
    normalize_hist = False,
)
fig.savefig(file_dir + "/angle_optimization_cc1pi.png", dpi=300)
plt.show()

# Michel removal selection

In [ ]:
cut_mask = (
    (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True) 
    & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    & (evt_df.slc.cut.angle == True)
)

bkg_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

signal_df = evt_df[
    cut_mask & 
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 11) | (abs(evt_df.pfp.trk.truth.p.pdg) == 22)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_num_hits  = ('pfp', 'max_daughter_hits', '', '', '', '')
col_mean_dEdx  = ('pfp', 'trk', 'mean_dEdx', '', '', '')
col_best_ke  = ('pfp', 'trk', 'calo', 'best', 'ke', '')
col_trk_score  = ('pfp', 'trackScore', '', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < 20) & (signal_df[col_chi2_p] > 85)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < 20) & (bkg_df[col_chi2_p] > 85)]
signal_df = signal_df[(signal_df[col_num_hits] == 0)]
bkg_df    = bkg_df[bkg_df[col_num_hits] ==0]
signal_df = signal_df[(signal_df[col_len] > 10)]
bkg_df    = bkg_df[bkg_df[col_len] > 10]

best_3D, results_df = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_mean_dEdx, col_best_ke,col_trk_score],
    cut_sign=["<", "<" , "<"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[0, 0, 0.5],               # min scan range
    cut_max=[10, 100, 0.7],            # max scan range
    n_steps= [11,21, 5],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)

best_2D, results_df = OptimizationUtils.manual_cut_optimizer(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cut_columns=[col_best_ke,col_trk_score],
    cut_sign=[ "<" , "<"],          # "<" for chi2_mu, ">" for chi2_proton
    cut_min=[ 0, 0.5],               # min scan range
    cut_max=[ 100, 0.7],            # max scan range
    n_steps= [101, 21],
    min_eff=0.0,
    min_pur=0.0,  # <-- new parameter                 # number of steps in each variable
)


In [ ]:

print("OG")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_mean_dEdx, col_best_ke,col_trk_score],
    cuts=[2.75,50,0.60],
    cut_sign=["<", "<" , "<"], 
)

ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = ["e/photon",r"$\mu/\pi$"],
    y_labels = ["e/photon",r"$\mu/\pi$"],
    cmap=sunset_cmap
)


print("3D")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_mean_dEdx, col_best_ke,col_trk_score],
    cuts=best_3D["cuts"],
    cut_sign=["<", "<" , "<"], 
)

ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = ["e/photon",r"$\mu/\pi$"],
    y_labels = ["e/photon",r"$\mu/\pi$"],
    cmap=sunset_cmap
)

print("2D no meandEdx")
cm = ConfusionMatricesUtils.build_confusion_matrix_from_cuts(
    signal_df=signal_df,
    bkg_df=bkg_df,
    cols=[col_mean_dEdx, col_best_ke,col_trk_score],
    cuts=[10,45, 0.6],
    cut_sign=["<", "<" , "<"], 
)


fig = ConfusionMatricesUtils.plot_confusion_matrix(
    cm = cm,
    x_labels = ["e/photon",r"$\mu/\pi$"],
    y_labels = ["e/photon",r"$\mu/\pi$"],
    cmap=sunset_cmap
)
fig.savefig(file_dir + "/confusion_matrix_michel_removal.png", dpi=300)
plt.show()




In [ ]:
# Split signal and background
cut_mask = (
    (evt_df.slc.cut.nu_score == True) & (evt_df.slc.cut.t0 == True) 
    & (evt_df.slc.cut.track == True) & (evt_df.slc.cut.inside_FV == True) 
    & (evt_df.slc.cut.MIP_candidates == True) & (evt_df.slc.cut.shower == True)
    & (evt_df.slc.cut.angle == True)
)

bkg_df = evt_df[
    cut_mask &
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 211) | (abs(evt_df.pfp.trk.truth.p.pdg) == 13)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

signal_df = evt_df[
    cut_mask & 
    ((abs(evt_df.pfp.trk.truth.p.pdg) == 11) | (abs(evt_df.pfp.trk.truth.p.pdg) == 22)) &
    (abs(evt_df.pfp.trk.len) > 3) & (abs(evt_df.pfp.trackScore) > 0.5) & (evt_df.pfp.dist_to_vertex < 10)
]

# -------------------------
# Columns
# -------------------------
col_chi2_mu = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', '')
col_chi2_p  = ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', '')
col_len  = ('pfp', 'trk', 'len', '', '', '')

col_num_hits  = ('pfp', 'max_daughter_hits', '', '', '', '')
col_mean_dEdx  = ('pfp', 'trk', 'mean_dEdx', '', '', '')
col_best_ke  = ('pfp', 'trk', 'calo', 'best', 'ke', '')
col_trk_score  = ('pfp', 'trackScore', '', '', '', '')


# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_chi2_mu] > 0) & (signal_df[col_chi2_p] > 0)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] > 0) & (bkg_df[col_chi2_p] > 0)]
signal_df = signal_df[(signal_df[col_chi2_mu] < 20) & (signal_df[col_chi2_p] > 85)]
bkg_df    = bkg_df[(bkg_df[col_chi2_mu] < 20) & (bkg_df[col_chi2_p] > 85)]
signal_df = signal_df[(signal_df[col_num_hits] == 0)]
bkg_df    = bkg_df[bkg_df[col_num_hits] ==0]
signal_df = signal_df[(signal_df[col_len] > 10)]
bkg_df    = bkg_df[bkg_df[col_len] > 10]

# -------------------------
# Remove rows with NaN or <=0 in the columns of interest
# -------------------------
signal_df = signal_df[(signal_df[col_trk_score] < 0.6) & (signal_df[col_best_ke] < 45)]
bkg_df    = bkg_df[(bkg_df[col_trk_score] < 0.6) & (bkg_df[col_best_ke] < 45)]

df_opt_len, best_len, fig = OptimizationUtils.optimize_cut_eff_pur(
    signal_df,
    bkg_df,
    column= col_mean_dEdx,
    cut_type="<",
    xlabel="dEdx [MeV/cm]",
    title="pfp optimization",
    signal_name=r"$\gamma/e$",
    bkg_name=r"$\mu/\pi$",
    xlim=(0, 10),
    nbins=51,
    legend_loc="upper right",
    cut_unit="[MeV/cm]",
    normalize_hist = False,
    min_pur=0.0 # <-- new parameter
)